# 第2回 宿題（発展）: 用語集の Q&A で LoRA 学習し、学習前後と用語集検索を比べる

演習と同じモデル・revision・LoRA 設定・生成設定で、学習データだけを FAQ から **用語集の Q&A（`glossary_sft.jsonl`）** に替えます。

**使うファイル（左のファイル一覧の `/content` にアップロード）**

| ファイル | 中身 |
| --- | --- |
| `glossary_sft.jsonl` | `scripts/make_sft_dataset.py` の出力。用語ごとに 2 つの質問テンプレート（messages 形式） |
| `glossary_eval.json` | 評価セット 26 問（用語集にある語 10・言い換え 8・用語集に無い語 8）と正解の条件 |

**評価**

- 用語集にある語・言い換え: 回答に、定義の重要語（`must_include`）がすべて含まれれば ○
- 用語集に無い語: 回答に「用語集」「載って」「わかりません」などの語（`refusal_markers`）があれば ○（知らないと言えたか）
- 全問で、回答に最も近い用語集の定義（`closest_term`）と類似度を記録し、別の用語の定義を流用していないかを見る

ランタイムは T4 GPU を選び、上から順に実行してください（合計 20〜25 分）。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
PINS = {"transformers": "5.18.0", "peft": "0.21.1", "trl": "1.14.2", "accelerate": "1.15.0", "datasets": "5.1.0"}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao
%pip install -q {spec}
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

In [ ]:
from pathlib import Path
import os, json, time, difflib

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 5,
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 160,
    "gen": {"do_sample": False},
    "data": "glossary_sft.jsonl",
    "work_dir": "/content/glossary_lab",
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

In [ ]:
needed = ["glossary_sft.jsonl", "glossary_eval.json"]
missing = [n for n in needed if not Path("/content", n).exists()]
if missing:
    from google.colab import files
    print("アップロードしてください:", missing)
    for name, data in files.upload().items():
        Path("/content", name).write_bytes(data)

records = [json.loads(line) for line in Path("/content/glossary_sft.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
EVAL = json.loads(Path("/content/glossary_eval.json").read_text(encoding="utf-8"))
items, REFUSAL = EVAL["items"], EVAL["refusal_markers"]

train_questions = {r["messages"][0]["content"] for r in records}
assert not [it["question"] for it in items if it["question"] in train_questions], "評価の質問が学習データに含まれています"
DEFINITIONS = {}
for r in records:
    q, a = r["messages"][0]["content"], r["messages"][1]["content"]
    if q.startswith("「"):
        DEFINITIONS[q[1:q.index("」")]] = a
print(f"学習 {len(records)} 件（用語 {len(DEFINITIONS)}）| 評価 {len(items)} 問")
print("学習例:", records[0]["messages"])

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
print("loaded:", CONFIG["base_model"], "| device:", DEVICE)

def generate(model, question):
    messages = [{"role": "user", "content": question}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)
GROUPS = ["in_glossary", "reworded", "not_in_glossary"]

def closest(answer):
    best = max(DEFINITIONS.items(), key=lambda kv: difflib.SequenceMatcher(None, answer, kv[1]).ratio())
    return best[0], round(difflib.SequenceMatcher(None, answer, best[1]).ratio(), 2)

def score(item, answer):
    if item["group"] == "not_in_glossary":
        return any(m in answer for m in REFUSAL)
    return all(w in answer for w in item["must_include"])

def evaluate(model, label):
    rows, t0 = [], time.time()
    for it in items:
        answer = generate(model, it["question"])
        term, sim = closest(answer)
        rows.append({"group": it["group"], "question": it["question"], "answer": answer, "ok": score(it, answer),
                     "refused": any(m in answer for m in REFUSAL), "closest_term": term, "similarity": sim})
    summary = {"label": label, "seconds": round(time.time() - t0, 1)}
    for g in GROUPS:
        g_rows = [r for r in rows if r["group"] == g]
        summary[g] = f"{sum(r['ok'] for r in g_rows)}/{len(g_rows)}"
    summary["all"] = f"{sum(r['ok'] for r in rows)}/{len(rows)}"
    (WORK / f"eval_glossary_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
display(pd.DataFrame([{"群": r["group"], "質問": r["question"], "学習前の回答": r["answer"][:160], "○×": "○" if r["ok"] else "×"} for r in before_rows]))

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(records)
model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=20, save_total_limit=2,
                max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
train_out = trainer.train()
TRAIN_SECONDS = round(time.time() - t0, 1)
losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
TRAIN_INFO = {"seconds": TRAIN_SECONDS, "steps": trainer.state.global_step, "epochs": round(trainer.state.epoch, 1),
              "first_loss": losses[0] if losses else None, "final_loss": losses[-1] if losses else None,
              "mean_train_loss": round(train_out.training_loss, 4)}
print(TRAIN_INFO)
trainer.model.save_pretrained(WORK / "adapter")
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()

In [ ]:
after_summary, after_rows = evaluate(tuned, "after")
display(pd.DataFrame([before_summary, after_summary]).set_index("label"))
display(pd.DataFrame([{"群": b["group"], "質問": b["question"],
                       "学習前": ("○ " if b["ok"] else "× ") + b["answer"][:120],
                       "学習後": ("○ " if a["ok"] else "× ") + a["answer"][:120],
                       "学習後に最も近い定義": f"{a['closest_term']} ({a['similarity']})"} for b, a in zip(before_rows, after_rows)]))

In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k != "work_dir"},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）・用語集 Q&A",
    "train": TRAIN_INFO, "train_records": len(records),
    "before": before_summary, "after": after_summary,
    "rows": [{"group": b["group"], "question": b["question"],
              "before": {k: b[k] for k in ("answer", "ok", "refused", "closest_term", "similarity")},
              "after": {k: a[k] for k in ("answer", "ok", "refused", "closest_term", "similarity")}}
             for b, a in zip(before_rows, after_rows)],
    "executed_training_myself": True,
}
(WORK / "result_glossary.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "rows"}, ensure_ascii=False, indent=2))
from google.colab import files
files.download(str(WORK / "result_glossary.json"))